# Qwen Baseline Pipeline — Qwen3.6-27B

Full 4-stage pipeline using **Qwen3.6-27B** via vLLM server. No OpenAI.

| Stage | Model |
|---|---|
| Router | Qwen3.6-27B |
| Analyze | Qwen3.6-27B |
| Extract | Qwen3.6-27B |
| Justify | Qwen3.6-27B |

Parameters from guide: `temperature=0.7`, `top_p=0.8`, `enable_thinking=False`

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
import sys, os, json, time, copy
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from dotenv import load_dotenv
load_dotenv(Path('/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/.env'), override=True)

from openai import OpenAI
import instructor

PROJECT_ROOT = Path('/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch')
FINAL_FOLDER = PROJECT_ROOT / 'FINAL FOLDER'
os.chdir(PROJECT_ROOT)
for p in [str(PROJECT_ROOT), str(FINAL_FOLDER)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from retrieve_data import retrieve_data
from prompts.default import (
    SYSTEM_PROMPT, DATA_ANALYSIS_PROMPT,
    CHART_CONFIGURATION_PROMPT, CREATE_CHART_TYPE_JUSTIFICATION_PROMPT,
)
from response_models.default import VisualizationConfig, DataAnalysis, ChartTypeJustification, ChartType
from visualization_from_template import generate_from_template

# ── Server & model (from guide) ────────────────────────────────────────────────
VLLM_URL   = 'http://hal9000.skim.th-owl.de:11877/v1'
QWEN_MODEL = 'Qwen3.6-27B'

qwen_raw   = OpenAI(base_url=VLLM_URL, api_key='dummy', timeout=300)
qwen_instr = instructor.from_openai(qwen_raw, mode=instructor.Mode.JSON)

# Parameters from guide — applied to every call
QWEN_PARAMS = dict(
    temperature=0.7,
    top_p=0.8,
    presence_penalty=1.5,
    extra_body={
        'top_k': 20,
        'chat_template_kwargs': {'enable_thinking': False},
    }
)

try:
    available = [m.id for m in qwen_raw.models.list().data]
    print(f'Server reachable. Models: {available}')
except Exception as e:
    print(f'Server check failed (run from university network): {e}')

MD_TABLE = retrieve_data(None, type='test')
QUERY    = 'Wieviel Umsatz hatte Teckentrup im Bereich JVA/CO in den Jahren 2021 bis 2024?'
N_RUNS   = 5

RUN_TS  = datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_DIR = FINAL_FOLDER / f'qwen_baseline_{RUN_TS}'
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'renders').mkdir(exist_ok=True)

ROUTER_PREVIEW = 500
ROUTER_TOOLS = [{
    'type': 'function',
    'function': {
        'name': 'generate_visualization',
        'description': 'Analyze the data and generate a visualization',
        'parameters': {
            'type': 'object',
            'properties': {
                'data':       {'type': 'string', 'description': 'The data as a markdown table'},
                'user_query': {'type': 'string', 'description': "The user's query"},
            },
            'required': ['data', 'user_query'],
        },
    },
}]

print(f'Model  : {QWEN_MODEL}')
print(f'Server : {VLLM_URL}')
print(f'Params : temp=0.7  top_p=0.8  top_k=20  enable_thinking=False')
print(f'Query  : {QUERY}')
print(f'Output : {OUT_DIR.name}')


In [ ]:
# ── Cell 2: Pipeline functions ─────────────────────────────────────────────────

def _usage(resp):
    u = getattr(resp, 'usage', None)
    if not u: return {}
    return {
        'prompt_tokens':     getattr(u, 'prompt_tokens', 0) or 0,
        'completion_tokens': getattr(u, 'completion_tokens', 0) or 0,
        'total_tokens':      getattr(u, 'total_tokens', 0) or 0,
    }

def step_router(query, md_table):
    t0 = time.perf_counter()
    resp = qwen_raw.chat.completions.create(
        model=QWEN_MODEL,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': f"{query}\n\nData preview:\n{md_table[:ROUTER_PREVIEW]}"},
        ],
        tools=ROUTER_TOOLS,
        tool_choice='auto',
        **QWEN_PARAMS,
    )
    return time.perf_counter() - t0, _usage(resp)

def step_analyze(query, md_table):
    prompt = DATA_ANALYSIS_PROMPT.format(data=md_table, query=query)
    t0 = time.perf_counter()
    resp, completion = qwen_instr.chat.completions.create_with_completion(
        model=QWEN_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        response_model=DataAnalysis,
        **QWEN_PARAMS,
    )
    return resp.analysis, time.perf_counter() - t0, _usage(completion)

def step_extract(analysis, md_table):
    prompt = CHART_CONFIGURATION_PROMPT.format(data=md_table, analysis=analysis)
    t0 = time.perf_counter()
    resp, completion = qwen_instr.chat.completions.create_with_completion(
        model=QWEN_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        response_model=VisualizationConfig,
        **QWEN_PARAMS,
    )
    cfg = resp.model_dump()
    cfg['charttype'] = cfg['charttype'].value if hasattr(cfg['charttype'], 'value') else str(cfg['charttype'])
    return cfg, time.perf_counter() - t0, _usage(completion)

def step_justify(analysis, md_table):
    charttypes = {ct.name for ct in ChartType}
    prompt = CREATE_CHART_TYPE_JUSTIFICATION_PROMPT.format(
        charttypes=charttypes, analysis=analysis, data=md_table)
    t0 = time.perf_counter()
    resp, completion = qwen_instr.chat.completions.create_with_completion(
        model=QWEN_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        response_model=ChartTypeJustification,
        **QWEN_PARAMS,
    )
    ct = resp.chart_type.value if hasattr(resp.chart_type, 'value') else str(resp.chart_type)
    return ct, time.perf_counter() - t0, _usage(completion)

def run_qwen_pipeline(query, md_table):
    timings, tokens = {}, {}
    t, tok = step_router(query, md_table)
    timings['router'] = t
    for k, v in tok.items(): tokens[k] = tokens.get(k, 0) + v

    analysis, t, tok = step_analyze(query, md_table)
    timings['analyze'] = t
    for k, v in tok.items(): tokens[k] = tokens.get(k, 0) + v

    cfg, t, tok = step_extract(analysis, md_table)
    timings['extract'] = t
    for k, v in tok.items(): tokens[k] = tokens.get(k, 0) + v

    charttype, t, tok = step_justify(analysis, md_table)
    cfg['charttype'] = charttype
    timings['justify'] = t
    for k, v in tok.items(): tokens[k] = tokens.get(k, 0) + v

    return cfg, timings, tokens

print(f'Pipeline ready — {QWEN_MODEL}  (enable_thinking=False)')


In [ ]:
# ── Cell 3: Quality validator ──────────────────────────────────────────────────
VALID_CT = {'line', 'bar', 'pie', 'scatter', 'column', 'stackedbar'}

def validate(cfg, render=True, tag='run', save_dir=None):
    r = {
        'schema_complete': False, 'valid_charttype': False, 'has_data': False,
        'has_title': False, 'has_axes': False, 'renderable': False,
        'has_annotations': False, 'data_consistency': False,
        'quality_score': 0, 'passed': False, 'error': None,
        'n_data_groups': 0, 'n_annotations': 0, 'title_text': '', 'charttype': ''
    }
    if cfg is None: r['error'] = 'null config'; return r
    if hasattr(cfg, 'model_dump'): cfg = cfg.model_dump()
    r['schema_complete'] = all(cfg.get(f) for f in ['titlename','charttype','xlabel','ylabel','data'])
    ct = cfg.get('charttype', '')
    if hasattr(ct, 'value'): ct = ct.value
    r['valid_charttype'] = str(ct).lower().strip() in VALID_CT
    r['charttype'] = str(ct)
    data = cfg.get('data') or []
    r['has_data'] = len(data) > 0
    r['n_data_groups'] = len(data)
    r['has_title'] = bool(str(cfg.get('titlename', '')).strip())
    r['title_text'] = str(cfg.get('titlename', ''))[:60]
    r['has_axes'] = bool(cfg.get('xlabel')) and bool(cfg.get('ylabel'))
    consistent = True
    for g in data:
        d = g if isinstance(g, dict) else (g.model_dump() if hasattr(g, 'model_dump') else {})
        xd, yd = d.get('x_data', []) or [], d.get('y_data', []) or []
        if not xd or not yd or len(xd) != len(yd): consistent = False; break
    r['data_consistency'] = consistent
    annos = cfg.get('annotations') or []
    r['has_annotations'] = len(annos) > 0
    r['n_annotations'] = len(annos)
    if render and r['has_data'] and r['valid_charttype']:
        try:
            rc = copy.deepcopy(cfg)
            xt = rc.get('x_ticks',[]) or []; xl = rc.get('x_tick_label',[]) or []
            if len(xl)!=len(xt): n=min(len(xt),len(xl)); rc['x_ticks']=xt[:n]; rc['x_tick_label']=xl[:n]
            yt = rc.get('y_ticks',[]) or []; yl = rc.get('y_tick_label',[]) or []
            if len(yl)!=len(yt): n=min(len(yt),len(yl)); rc['y_ticks']=yt[:n]; rc['y_tick_label']=yl[:n]
            _show = plt.show; plt.show = lambda *a, **k: None
            generate_from_template(rc)
            if save_dir: plt.savefig(Path(save_dir)/f'{tag}.png', dpi=120, bbox_inches='tight')
            plt.show = _show; plt.close('all')
            r['renderable'] = True
        except Exception as e:
            r['error'] = f'render: {str(e)[:80]}'
            plt.close('all')
            try: plt.show = _show
            except: pass
    checks = ['schema_complete','valid_charttype','has_data','has_title',
               'has_axes','renderable','has_annotations','data_consistency']
    r['quality_score'] = sum(r[c] for c in checks)
    r['passed'] = r['quality_score'] >= 5
    return r

print('Quality validator ready (8 checks)')


In [ ]:
# ── Cell 4: Run N iterations ──────────────────────────────────────────────────
print(f'Running {N_RUNS} iterations — {QWEN_MODEL}')
print(f'Query: {QUERY}\n')

results = []

for i in range(1, N_RUNS + 1):
    label = 'COLD' if i == 1 else 'warm'
    print('='*60)
    print(f'Run {i}/{N_RUNS}  [{label}]')
    t0 = time.perf_counter()
    try:
        cfg, timings, tokens = run_qwen_pipeline(QUERY, MD_TABLE)
        total = time.perf_counter() - t0
        q = validate(cfg, render=True, tag=f'run{i}', save_dir=OUT_DIR/'renders')
        err = None
    except Exception as e:
        total = time.perf_counter() - t0
        cfg, timings, tokens = None, {}, {}
        q = {'quality_score':0,'passed':False,'charttype':'error','title_text':'',
             'n_data_groups':0,'n_annotations':0,'renderable':False}
        err = str(e)[:150]
        print(f'  ERROR: {err}')

    print(f'  Total   : {total:.2f}s')
    for stage, t in timings.items(): print(f'  {stage:<10}: {t:.2f}s')
    print(f'  Quality : {q["quality_score"]}/8  passed={q["passed"]}  chart={q["charttype"]}')

    results.append({
        'run': i, 'cold_start': i==1, 'model': QWEN_MODEL,
        'total_latency':   round(total, 3),
        'router_latency':  round(timings.get('router', 0), 3),
        'analyze_latency': round(timings.get('analyze', 0), 3),
        'extract_latency': round(timings.get('extract', 0), 3),
        'justify_latency': round(timings.get('justify', 0), 3),
        'quality_score':   q['quality_score'], 'passed': q['passed'],
        'charttype':       q['charttype'], 'title': q['title_text'],
        'n_data_groups':   q['n_data_groups'], 'n_annotations': q['n_annotations'],
        'renderable':      q['renderable'],
        'tokens_total':    tokens.get('total_tokens', 0),
        'error':           err or '',
    })

print('\n' + '='*60)
print('Done.')


In [ ]:
# ── Cell 5: Summary + statistics + CSV ────────────────────────────────────────
df = pd.DataFrame(results)

# ── Per-run table ──────────────────────────────────────────────────────────────
print('\n' + '='*75)
print(f'  QWEN BASELINE — {QWEN_MODEL}  ({N_RUNS} runs)')
print('='*75)
print(f'  {"Run":<5} {"Cold":>5}  {"Total":>7}  {"Router":>7}  {"Analyze":>8}  {"Extract":>8}  {"Justify":>8}  {"Q":>4}')
print('  ' + '-'*73)
for _, r in df.iterrows():
    cold = 'COLD' if r['cold_start'] else 'warm'
    print(f"  {int(r['run']):<5} {cold:>5}  {r['total_latency']:>6.2f}s  "
          f"{r['router_latency']:>6.2f}s  {r['analyze_latency']:>7.2f}s  "
          f"{r['extract_latency']:>7.2f}s  {r['justify_latency']:>7.2f}s  "
          f"{r['quality_score']:>2}/8")

# ── Statistics on warm runs (exclude run 1 cold start) ────────────────────────
warm = df[~df['cold_start']]['total_latency']
all_runs = df['total_latency']

def stats(series, label):
    s = series.values
    print(f'\n  {label} (n={len(s)}):')
    print(f'    Mean   : {s.mean():.2f}s')
    print(f'    Median : {float(np.median(s)):.2f}s')
    print(f'    Std Dev: {s.std():.2f}s')
    print(f'    P90    : {float(np.percentile(s, 90)):.2f}s')
    print(f'    P95    : {float(np.percentile(s, 95)):.2f}s')
    print(f'    P99    : {float(np.percentile(s, 99)):.2f}s')
    print(f'    Min    : {s.min():.2f}s')
    print(f'    Max    : {s.max():.2f}s')

print('\n' + '='*75)
stats(all_runs, 'All runs (total latency)')
if len(warm) > 0:
    stats(warm, 'Warm runs only (excluding cold start)')

# ── Per-stage stats (all runs) ────────────────────────────────────────────────
print('\n  Per-stage mean latency:')
for stage in ['router_latency', 'analyze_latency', 'extract_latency', 'justify_latency']:
    vals = df[stage].values
    print(f'    {stage:<20}: mean={vals.mean():.2f}s  median={float(np.median(vals)):.2f}s  '
          f'p95={float(np.percentile(vals,95)):.2f}s')

print(f'\n  Quality avg : {df["quality_score"].mean():.1f}/8')
print(f'  All passed  : {df["passed"].all()}')
print('='*75)

# ── Build statistics dataframe ────────────────────────────────────────────────
stat_rows = []
for col in ['total_latency','router_latency','analyze_latency','extract_latency','justify_latency']:
    vals = df[col].values
    stat_rows.append({
        'metric': col,
        'mean':   round(vals.mean(), 3),
        'median': round(float(np.median(vals)), 3),
        'std':    round(vals.std(), 3),
        'p90':    round(float(np.percentile(vals, 90)), 3),
        'p95':    round(float(np.percentile(vals, 95)), 3),
        'p99':    round(float(np.percentile(vals, 99)), 3),
        'min':    round(vals.min(), 3),
        'max':    round(vals.max(), 3),
        'n':      len(vals),
    })
df_stats = pd.DataFrame(stat_rows)

csv_runs  = OUT_DIR / f'qwen_baseline_runs_{RUN_TS}.csv'
csv_stats = OUT_DIR / f'qwen_baseline_stats_{RUN_TS}.csv'
df.to_csv(csv_runs, index=False)
df_stats.to_csv(csv_stats, index=False)
print(f'\n  Runs CSV  : {csv_runs}')
print(f'  Stats CSV : {csv_stats}')


In [ ]:
# ── Cell 6: Latency chart (all runs + stats) ──────────────────────────────────
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={'width_ratios':[2,1]})

# Left: stacked bar per run
labels   = [f'Run {r}\n({"cold" if c else "warm"})' for r, c in zip(df['run'], df['cold_start'])]
routers  = df['router_latency'].tolist()
analyzes = df['analyze_latency'].tolist()
extracts = df['extract_latency'].tolist()
justifys = df['justify_latency'].tolist()
totals   = df['total_latency'].tolist()

stacks = [routers, analyzes, extracts, justifys]
colors = ['#4caf50','#ab47bc','#ff7043','#26c6da']
lbls   = ['Router','Analyze','Extract','Justify']
bottoms = [0.0]*len(labels)
for vals, col, lbl in zip(stacks, colors, lbls):
    ax.bar(range(len(labels)), vals, 0.5, bottom=bottoms, label=lbl, color=col)
    bottoms = [b+v for b,v in zip(bottoms, vals)]
for i, t in enumerate(totals):
    ax.text(i, t+0.3, f'{t:.1f}s', ha='center', va='bottom', fontsize=8, fontweight='bold', color='white')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel('Latency (s)', color='#aaa')
ax.set_title(f'{QWEN_MODEL} — Per-Run Breakdown', color='white', fontsize=10, pad=8)
ax.legend(loc='upper right', fontsize=7, framealpha=0.3)
ax.set_facecolor('#13131d'); ax.tick_params(colors='#aaa')
for sp in ax.spines.values(): sp.set_edgecolor('#2a2a3a')

# Right: stats summary box
t_vals = np.array(totals)
stat_labels = [
    f"Mean   {t_vals.mean():.2f}s",
    f"Median {float(np.median(t_vals)):.2f}s",
    f"Std    {t_vals.std():.2f}s",
    f"P90    {float(np.percentile(t_vals,90)):.2f}s",
    f"P95    {float(np.percentile(t_vals,95)):.2f}s",
    f"P99    {float(np.percentile(t_vals,99)):.2f}s",
    f"Min    {t_vals.min():.2f}s",
    f"Max    {t_vals.max():.2f}s",
]
ax2.axis('off')
ax2.set_facecolor('#13131d')
y = 0.95
for line in stat_labels:
    ax2.text(0.05, y, line, transform=ax2.transAxes,
             fontsize=10, color='white', family='monospace', va='top')
    y -= 0.11
ax2.set_title('Total Latency Stats', color='white', fontsize=10, pad=8)
ax2.set_facecolor('#13131d')

fig.patch.set_facecolor('#1a1a24')
plt.tight_layout()
chart_path = OUT_DIR / f'qwen_baseline_chart_{RUN_TS}.png'
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart: {chart_path}')
